In [0]:
df = spark.read.format("parquet")\
            .load("abfss://bronze@databricksetestorage2003.dfs.core.windows.net/data/orders")
display(df)

In [0]:
df.printSchema()

In [0]:
print(df.columns)

In [0]:
df = df.drop('_rescued_data')

In [0]:
from pyspark.sql.functions import to_timestamp, col; 

df = df.withColumn('order_date', to_timestamp(col('order_date')))


In [0]:
display(df)

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("year", year(col('order_date')))


In [0]:
display(df)

In [0]:
from pyspark.sql.window import Window

df1 = df.withColumn('flag', dense_rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))      
display(df1)

In [0]:
class windows:
    
    def dense_rank(self,df):
        df_dense_rank = df.withColumn("dense_rank", dense_rank().over(Window.partitionBy('year')).orderBy(desc('total_amount')))
        return df_dense_rank
    
    def row_number(self,df):
        df_row_number = df.withColumn("row_number", row_number().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_row_number
    
    def rank(self,df):
        df_rank = df.withColumn("rank", rank().over(Window.partitionBy('year').orderBy(desc('total_amount'))))
        return df_rank
    


In [0]:
obj = windows()

df_results = obj.dense_rank(df)

display(df_results)

In [0]:

df.write.format("delta").mode("append").save("abfss://silver@databricksetestorage2003.dfs.core.windows.net/orders")